# Prithvi-EO-2.0 + Sen1Floods11 BWER Paper-Prep Workflow

This notebook runs the existing Prithvi + Sen1Floods11 pipeline, then applies slice-support preflight and formal BWER audit. It supports a 64-sample smoke mode and a full or near-full paper-prep mode.

Default mode is `PAPER_PREP_FULL=True`, intended for Colab Pro High-RAM. For a quick check, set `PAPER_PREP_FULL=False` and `SMOKE_MODE=True` in the configuration cell.


In [ ]:
# 1. Clone or update this repo
from pathlib import Path
import os
REPO_URL = "https://github.com/strivekboy-coder/rsfm-fairness-audit.git"  # edit if you use a fork
PROJECT_ROOT = Path("/content/rsfm-fairness-audit")
%cd /content
if PROJECT_ROOT.exists():
    %cd {PROJECT_ROOT}
    !git pull
else:
    !git clone {REPO_URL} {PROJECT_ROOT}
    %cd {PROJECT_ROOT}
os.chdir(PROJECT_ROOT)
print('repo root:', Path.cwd())


In [ ]:
# 2. Install package and Prithvi dependencies
# Keep numpy below 2.1 because Colab's numba stack requires numpy<2.1.
!python -m pip install --upgrade --force-reinstall pandas==2.2.2 "numpy>=1.24,<2.1"
!python -m pip install -e .
!python -m pip install -r requirements-prithvi.txt
!python -m pip install --upgrade terratorch "numpy>=1.24,<2.1"
!python -c "import numpy, pandas; print('numpy', numpy.__version__); print('pandas', pandas.__version__); import importlib.util; spec = importlib.util.find_spec('numba'); print('numba', __import__('numba').__version__ if spec else 'not installed')"


In [ ]:
# 3. Check GPU
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
# 4. Configure smoke/full mode and run paths
PAPER_PREP_FULL = True
SMOKE_MODE = not PAPER_PREP_FULL
RUN_CLASSIFICATION = True
RUN_SEGMENTATION = False  # enable after classification+BWER works; segmentation is heavier
LOW_RAM_MODE = False if PAPER_PREP_FULL else True

MAX_SAMPLES = 100000 if PAPER_PREP_FULL else 64
CANDIDATE_LIMIT = 100000 if PAPER_PREP_FULL else 1000
CHUNK_SIZE = 16 if PAPER_PREP_FULL else 8
BATCH_SIZE = 1  # safest for Prithvi on Colab; raise to 2 on High-RAM if stable
BOOTSTRAP_N = 100 if PAPER_PREP_FULL else 50
MIN_SAMPLES_PER_SLICE = 20 if PAPER_PREP_FULL else 1
MIN_SLICES_REQUIRED = 2

RUN_TAG = "full" if PAPER_PREP_FULL else f"smoke{MAX_SAMPLES}"
DATA_ROOT = f"data/sen1floods11_prithvi_subset_{RUN_TAG}"
CLASS_OUTPUT = f"outputs/prithvi_sen1floods11_class_{RUN_TAG}"
SEG_OUTPUT = f"outputs/prithvi_sen1floods11_seg_{RUN_TAG}"
AUDIT_ROOT = f"outputs/audit/prithvi_sen1floods11_bwer_{RUN_TAG}"
SUPPORT_ROOT = f"{AUDIT_ROOT}/metadata_support"
CLASS_PREFLIGHT = f"{AUDIT_ROOT}/classification_preflight"
SEG_PREFLIGHT = f"{AUDIT_ROOT}/segmentation_preflight"
CLASS_AUDIT_TABLE = f"{AUDIT_ROOT}/classification_audit_table.csv"
SEG_AUDIT_TABLE = f"{AUDIT_ROOT}/segmentation_audit_table.csv"
FORMAL_ROOT = f"{AUDIT_ROOT}/formal_bwer"

print(DATA_ROOT, CLASS_OUTPUT, SEG_OUTPUT, AUDIT_ROOT, sep='\n')
print('PAPER_PREP_FULL', PAPER_PREP_FULL, 'SMOKE_MODE', SMOKE_MODE)
print('RUN_CLASSIFICATION', RUN_CLASSIFICATION, 'RUN_SEGMENTATION', RUN_SEGMENTATION)
print('CHUNK_SIZE', CHUNK_SIZE, 'BATCH_SIZE', BATCH_SIZE, 'BOOTSTRAP_N', BOOTSTRAP_N)


In [ ]:
# 5. Apply notebook batch-size control to the Prithvi config
from pathlib import Path
import yaml
config_path = Path("configs/models/prithvi.yaml")
config = yaml.safe_load(config_path.read_text())
config["batch_size"] = int(BATCH_SIZE)
config["device"] = "auto"
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path.read_text())


In [ ]:
# 6. Prepare Sen1Floods11 samples directly in Colab local storage
!python scripts/prepare_sen1floods11_subset.py   --output-dir {DATA_ROOT}   --max-samples {MAX_SAMPLES}   --candidate-limit {CANDIDATE_LIMIT}
!head -5 {DATA_ROOT}/metadata.csv


## Metadata Support Gate Before Prithvi Inference

Before loading Prithvi, the notebook checks whether the prepared Sen1Floods11 metadata has enough event-level support for formal BWER. This avoids spending GPU/RAM on a run that can only produce invalid tail-risk estimates.


In [ ]:
# 7. Metadata support report and paper-prep gate
from pathlib import Path
import pandas as pd

support_dir = Path(SUPPORT_ROOT)
support_dir.mkdir(parents=True, exist_ok=True)
metadata = pd.read_csv(Path(DATA_ROOT) / "metadata.csv")
metadata["event_id"] = metadata.get("event_id", metadata.get("event", metadata.get("region", "to_verify"))).astype(str)
metadata["class_label"] = metadata.get("class_label", metadata["label"].astype(str)).astype(str)
metadata["flood_label"] = metadata.get("flood_label", metadata["label"].astype(str)).astype(str)

event_counts = metadata["event_id"].value_counts().rename_axis("event_id").reset_index(name="n")
class_counts = metadata["class_label"].value_counts().rename_axis("class_label").reset_index(name="n")
flood_counts = metadata["flood_label"].value_counts().rename_axis("flood_label").reset_index(name="n")
event_class = metadata.groupby(["event_id", "class_label"]).size().reset_index(name="n")
event_flood = metadata.groupby(["event_id", "flood_label"]).size().reset_index(name="n")

event_counts.to_csv(support_dir / "event_id_counts.csv", index=False)
class_counts.to_csv(support_dir / "class_label_counts.csv", index=False)
flood_counts.to_csv(support_dir / "flood_label_counts.csv", index=False)
event_class.to_csv(support_dir / "event_id_by_class_label_support.csv", index=False)
event_flood.to_csv(support_dir / "event_id_by_flood_label_support.csv", index=False)

country_counts = None
if "country" in metadata.columns:
    country_counts = metadata["country"].value_counts().rename_axis("country").reset_index(name="n")
    country_counts.to_csv(support_dir / "country_counts.csv", index=False)

valid_events = event_counts[event_counts["n"] >= MIN_SAMPLES_PER_SLICE]
report_lines = [
    "# Sen1Floods11 Metadata Support Report",
    "",
    f"- Prepared samples: {len(metadata)}",
    f"- Event slices total: {len(event_counts)}",
    f"- Event slices with n >= {MIN_SAMPLES_PER_SLICE}: {len(valid_events)}",
    f"- Minimum slices required: {MIN_SLICES_REQUIRED}",
    "",
    "## Event Counts",
    "```",
    event_counts.to_string(index=False),
    "```",
    "",
    "## Class Counts",
    "```",
    class_counts.to_string(index=False),
    "```",
]
if country_counts is not None:
    report_lines.extend(["", "## Country Counts", "```", country_counts.to_string(index=False), "```"])
(support_dir / "metadata_support_report.md").write_text("\n".join(report_lines) + "\n", encoding="utf-8")

display(event_counts)
display(class_counts)
display(event_class.head(20))

if PAPER_PREP_FULL and len(valid_events) < MIN_SLICES_REQUIRED:
    raise RuntimeError(
        f"Stopping before Prithvi inference: only {len(valid_events)} event_id slices have at least "
        f"{MIN_SAMPLES_PER_SLICE} samples. Increase candidate_limit/use more data or treat this as smoke only."
    )


In [ ]:
# 8. Existing real-run preflight
!python -m rsfm_fairness_audit.cli check-real   --dataset sen1floods11   --model prithvi   --model-config configs/models/prithvi.yaml   --data-root {DATA_ROOT}


In [ ]:
# 9. Chip-level classification sanity audit
import gc, subprocess, torch

if RUN_CLASSIFICATION:
    cmd = [
        "python", "-m", "rsfm_fairness_audit.cli", "run-real",
        "--dataset", "sen1floods11",
        "--model", "prithvi",
        "--dataset-root", DATA_ROOT,
        "--config", "configs/models/prithvi.yaml",
        "--output-dir", CLASS_OUTPUT,
        "--max-samples", str(MAX_SAMPLES),
        "--chunk-size", str(CHUNK_SIZE),
        "--streaming-embeddings", "true",
    ]
    print(" ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("Skipping classification because RUN_CLASSIFICATION=False")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# 10. Optional lightweight segmentation fairness audit
import gc, subprocess, torch

if RUN_SEGMENTATION:
    cmd = [
        "python", "-m", "rsfm_fairness_audit.cli", "run-segmentation-real",
        "--dataset", "sen1floods11",
        "--model", "prithvi",
        "--dataset-root", DATA_ROOT,
        "--config", "configs/models/prithvi.yaml",
        "--output-dir", SEG_OUTPUT,
        "--max-samples", str(MAX_SAMPLES),
    ]
    print(" ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("Skipping segmentation because RUN_SEGMENTATION=False")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## BWER Preflight

`preflight-bwer` checks whether each candidate has enough support before formal BWER. Sparse event x class support can make balanced BWER unstable, so formal BWER is run only for candidates marked runnable and recommended by default. If none are recommended, the notebook falls back to the best runnable caution candidates as pilot evidence.


In [ ]:
# 11. Build normalized audit tables and inspect available columns
from pathlib import Path
import pandas as pd
from rsfm_fairness_audit.audit_table import (
    build_audit_table_from_predictions,
    build_audit_table_from_segmentation_metrics,
    write_audit_table,
)

Path(AUDIT_ROOT).mkdir(parents=True, exist_ok=True)
metadata_path = Path(DATA_ROOT) / "metadata.csv"
class_rows = []
seg_rows = []
if RUN_CLASSIFICATION and (Path(CLASS_OUTPUT) / "predictions.csv").exists():
    class_rows = build_audit_table_from_predictions(
        Path(CLASS_OUTPUT) / "predictions.csv",
        metadata_path=metadata_path,
        dataset="sen1floods11",
        model="prithvi",
        task="classification",
    )
if RUN_SEGMENTATION and (Path(SEG_OUTPUT) / "segmentation_metrics.csv").exists():
    seg_rows = build_audit_table_from_segmentation_metrics(
        Path(SEG_OUTPUT) / "segmentation_metrics.csv",
        metadata_path=metadata_path,
        dataset="sen1floods11",
        model="prithvi",
        task="segmentation",
    )

def enrich_sen1_rows(rows, segmentation=False):
    enriched = []
    for row in rows:
        item = dict(row)
        if not item.get("event_id") and item.get("event"):
            item["event_id"] = item["event"]
        if segmentation:
            item.setdefault("class_label", "water")
            item.setdefault("flood_label", item.get("class_label", "water"))
        else:
            item.setdefault("flood_label", str(item.get("label", item.get("class_label", ""))))
        enriched.append(item)
    return enriched

class_rows = enrich_sen1_rows(class_rows, segmentation=False)
seg_rows = enrich_sen1_rows(seg_rows, segmentation=True)
if class_rows:
    write_audit_table(CLASS_AUDIT_TABLE, class_rows)
if seg_rows:
    write_audit_table(SEG_AUDIT_TABLE, seg_rows)

for name, path_value in [("classification", CLASS_AUDIT_TABLE), ("segmentation", SEG_AUDIT_TABLE)]:
    path = Path(path_value)
    if not path.exists():
        print("Skipping {}: {} not found".format(name, path))
        continue
    df = pd.read_csv(path)
    print("\n=== {} audit table: {} ===".format(name, path))
    print("rows", len(df))
    print("columns", list(df.columns))
    display(df.head())


In [ ]:
# 12. Choose candidate BWER configurations from available metadata
PRIORITY_CANDIDATES = [
    ("event_id", "class_label"),
    ("country", "class_label"),
    ("month", "class_label"),
    ("season", "class_label"),
    ("biome", "class_label"),
    ("ecoregion", "class_label"),
    ("event_id", "flood_label"),
    ("country", "flood_label"),
    ("event_id", None),
    ("country", None),
    ("season", None),
]

def available_candidates(csv_path):
    path = Path(csv_path)
    if not path.exists():
        return [], [("", "", "audit table missing")]
    df = pd.read_csv(path, nrows=1)
    columns = set(df.columns)
    out = []
    skipped = []
    for slice_var, balance_var in PRIORITY_CANDIDATES:
        if slice_var not in columns:
            skipped.append((slice_var, balance_var, "missing slice column"))
            continue
        if balance_var and balance_var not in columns:
            skipped.append((slice_var, balance_var, "missing balance column"))
            continue
        out.append(f"{slice_var}|{balance_var}" if balance_var else slice_var)
    return out, skipped

CLASS_CANDIDATES, CLASS_SKIPPED = available_candidates(CLASS_AUDIT_TABLE)
SEG_CANDIDATES, SEG_SKIPPED = available_candidates(SEG_AUDIT_TABLE)
print("classification candidates", CLASS_CANDIDATES)
print("classification skipped", CLASS_SKIPPED)
print("segmentation candidates", SEG_CANDIDATES)
print("segmentation skipped", SEG_SKIPPED)


In [ ]:
# 13. Run BWER support preflight
import subprocess
import pandas as pd

def run_preflight(audit_table, output_dir, candidates, task_name):
    if not candidates or not Path(audit_table).exists():
        print("Skipping {} preflight: no audit table or candidates".format(task_name))
        return
    cmd = [
        "python", "-m", "rsfm_fairness_audit.cli", "preflight-bwer",
        "--audit-table", str(audit_table),
        "--dataset", "sen1floods11",
        "--model", "prithvi",
        "--task", task_name,
        "--output-dir", str(output_dir),
        "--min-samples-per-slice", str(MIN_SAMPLES_PER_SLICE),
        "--min-units-required", str(MIN_SAMPLES_PER_SLICE),
        "--min-slices-required", str(MIN_SLICES_REQUIRED),
    ]
    for candidate in candidates:
        cmd.extend(["--candidate", candidate])
    print(" ".join(cmd))
    subprocess.check_call(cmd)

run_preflight(CLASS_AUDIT_TABLE, CLASS_PREFLIGHT, CLASS_CANDIDATES, "classification")
run_preflight(SEG_AUDIT_TABLE, SEG_PREFLIGHT, SEG_CANDIDATES, "segmentation")

for name, out_dir in [("classification", CLASS_PREFLIGHT), ("segmentation", SEG_PREFLIGHT)]:
    rec_path = Path(out_dir) / "slice_support_recommendations.csv"
    if rec_path.exists():
        print("\n=== {} support recommendations ===".format(name))
        recs = pd.read_csv(rec_path)
        display(recs[["candidate", "recommendation", "formal_bwer_runnable", "preferred_bwer", "n_slices_valid", "missing_slice_balance_ratio", "reason"]])


In [ ]:
# 14. Run formal BWER only for runnable recommended candidates; fall back to runnable caution candidates as pilot-only
import subprocess
import pandas as pd

CLASS_BWER_DIRS = []
SEG_BWER_DIRS = []

def parse_candidate(text):
    inner = text.replace("BWER(", "").rstrip(")")
    if " | " in inner:
        left, right = inner.split(" | ", 1)
        return left, right
    return inner, None

def select_candidates(preflight_dir, max_caution=2):
    rec_path = Path(preflight_dir) / "slice_support_recommendations.csv"
    if not rec_path.exists():
        return [], "pilot"
    recs = pd.read_csv(rec_path)
    runnable = recs[recs["formal_bwer_runnable"].astype(str).str.lower().isin(["true", "1"])]
    recommended = runnable[runnable["recommendation"] == "recommended"].copy()
    if len(recommended):
        return recommended["candidate"].tolist(), "paper"
    caution = runnable[runnable["recommendation"] == "caution"].copy()
    if len(caution):
        caution["_missing"] = pd.to_numeric(caution["missing_slice_balance_ratio"], errors="coerce").fillna(0.0)
        caution["_valid"] = pd.to_numeric(caution["n_slices_valid"], errors="coerce").fillna(0)
        caution = caution.sort_values(["_valid", "_missing"], ascending=[False, True]).head(max_caution)
        return caution["candidate"].tolist(), "pilot"
    return [], "pilot"

def run_formal_bwer(audit_table, preflight_dir, task_name):
    if not Path(audit_table).exists():
        print("Skipping formal BWER for {}: missing {}".format(task_name, audit_table))
        return []
    selected, level = select_candidates(preflight_dir)
    print("{}: selected {} as audit_level={}".format(task_name, selected, level))
    outputs = []
    for candidate in selected:
        slice_var, balance_var = parse_candidate(candidate)
        safe_name = candidate.replace("BWER(", "").replace(")", "").replace(" | ", "__").replace(" ", "_")
        out_dir = Path(FORMAL_ROOT) / task_name / safe_name
        cmd = [
            "python", "-m", "rsfm_fairness_audit.cli", "evaluate-bwer",
            "--audit-table", str(audit_table),
            "--dataset", "sen1floods11",
            "--model", "prithvi",
            "--task", task_name,
            "--slice-variable", slice_var,
            "--output-dir", str(out_dir),
            "--missing-balance-policy", "renormalize",
            "--bootstrap", str(BOOTSTRAP_N),
            "--audit-level", level,
        ]
        if balance_var:
            cmd.extend(["--balance-variable", balance_var])
        print(" ".join(cmd))
        subprocess.check_call(cmd)
        outputs.append(out_dir)
    if not outputs:
        print("No runnable recommended/caution candidates for {}; formal BWER skipped.".format(task_name))
    return outputs

CLASS_BWER_DIRS = run_formal_bwer(CLASS_AUDIT_TABLE, CLASS_PREFLIGHT, "classification")
SEG_BWER_DIRS = run_formal_bwer(SEG_AUDIT_TABLE, SEG_PREFLIGHT, "segmentation")


In [ ]:
# 15. Inspect BWER outputs and figures
from IPython.display import Image, display

for out_dir in CLASS_BWER_DIRS + SEG_BWER_DIRS:
    out_dir = Path(out_dir)
    print("\n=== {} ===".format(out_dir))
    for csv_name in ["bwer_summary.csv", "bwer_by_slice.csv", "support_diagnostics.csv", "bootstrap_ci.csv"]:
        path = out_dir / csv_name
        if path.exists() and path.stat().st_size > 0:
            print(csv_name)
            display(pd.read_csv(path).head(20))
    for fig_name in ["average_vs_bwer.png", "raw_vs_balanced_bwer.png", "worst_tail_slices.png", "slice_risk_heatmap.png"]:
        fig_path = out_dir / "figures" / fig_name
        if fig_path.exists():
            print(fig_name)
            display(Image(filename=str(fig_path)))


## Suggested Scientific Finding Note

After inspecting support and BWER outputs, add only a concise interpretation to `docs/experiments/scientific_findings.md`. Do not paste raw logs or full tables there.


In [ ]:
# 16. Optionally append a concise finding note to docs/experiments/scientific_findings.md
from datetime import date
finding_path = Path("docs/experiments/scientific_findings.md")
finding_path.parent.mkdir(parents=True, exist_ok=True)
summary_lines = [
    "",
    "## Prithvi Sen1Floods11 BWER Paper-Prep ({})".format(date.today().isoformat()),
    "",
    "A {}-sample Prithvi-EO-2.0 + Sen1Floods11 run was used to test metadata support, BWER support preflight, and formal audit wiring. Interpret results according to support recommendations; raw outputs remain in Colab output directories and are not stored in this findings log.".format(MAX_SAMPLES),
]
print("\n".join(summary_lines))
# Uncomment to append after review:
# with finding_path.open("a", encoding="utf-8") as handle:
#     handle.write("\n".join(summary_lines) + "\n")


In [ ]:
# 17. Package final BWER evidence artifacts for download
from zipfile import ZIP_DEFLATED, ZipFile
from google.colab import files

PROJECT_ROOT = Path('/content/rsfm-fairness-audit').resolve()
ZIP_PATH = PROJECT_ROOT / f"prithvi_sen1floods11_bwer_{RUN_TAG}_evidence.zip"
roots = [Path(AUDIT_ROOT), Path(CLASS_OUTPUT), Path(SEG_OUTPUT)]
include_names = {
    "audit_table.csv",
    "slice_support_recommendations.csv",
    "slice_support_summary.csv",
    "slice_support_report.md",
    "metadata_support_report.md",
    "event_id_counts.csv",
    "country_counts.csv",
    "class_label_counts.csv",
    "flood_label_counts.csv",
    "event_id_by_class_label_support.csv",
    "event_id_by_flood_label_support.csv",
    "warnings.json",
    "bwer_summary.csv",
    "bwer_by_slice.csv",
    "support_diagnostics.csv",
    "bootstrap_ci.csv",
    "report.md",
    "segmentation_metrics.csv",
    "fairness_summary.csv",
    "raw_vs_balanced_gap.csv",
}
files_to_zip = []
for root in roots:
    root = (PROJECT_ROOT / root).resolve() if not Path(root).is_absolute() else Path(root).resolve()
    if not root.exists():
        continue
    for path in root.rglob("*"):
        if path.is_file() and (path.name in include_names or "figures" in path.parts or "tables" in path.parts):
            files_to_zip.append(path.resolve())
with ZipFile(ZIP_PATH, "w", compression=ZIP_DEFLATED) as archive:
    for path in sorted(set(files_to_zip)):
        archive.write(path, path.relative_to(PROJECT_ROOT).as_posix())
print("Packaged {} artifacts into {}".format(len(files_to_zip), ZIP_PATH))
files.download(str(ZIP_PATH))
